# GRPO (Group Relative Policy Optimization) Loss

**难度:** Hard

实现 GRPO（组相对策略优化）损失。

GRPO 在每个提示组内归一化奖励以计算优势值，然后使用这些组相对优势优化策略。

**签名:** `grpo_loss(logps, rewards, group_ids, eps=1e-5) -> Tensor`

**参数:**
- `logps` — 策略对数概率 (B,)
- `rewards` — 标量奖励 (B,)
- `group_ids` — 整数组标识符 (B,)
- `eps` — 数值稳定性的 epsilon

**返回:** 标量损失

**约束:**
- 组内 z-score 归一化：`A_i = (r_i - mean_g) / (std_g + eps)`
- 优势值必须 detach（梯度仅通过 logps 流动）
- 损失 = `-mean(A_i * logps_i)`

**提示:**
1. 对每个组 g：A_i = (r_i - mean_g) / (std_g + eps)
2. advantages = A.detach()（梯度不流经奖励）
3. loss = -(advantages * logps).mean()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
# ✅ INTERVIEW

# ---- 手写 GRPO Loss ----
# 面试要点：理解组内归一化的优势 + detach 的作用

def grpo_loss(logps: Tensor, rewards: Tensor, group_ids: Tensor,
              eps: float = 1e-5) -> Tensor:
    # 面试考点：为什么需要组内归一化？
    # 答：不同 prompt 的奖励尺度不同（有的 prompt 天然高分）
    # 组内归一化后，优势值只反映"在这个 prompt 下，这个回复好不好"
    unique_ids = group_ids.unique()
    advantages = torch.empty_like(rewards)

    for gid in unique_ids:
        mask = group_ids == gid
        r_g = rewards[mask]
        mean_g = r_g.mean()
        std_g = r_g.std(unbiased=False)  # 面试考点：unbiased=False 用 N 做分母
        advantages[mask] = (r_g - mean_g) / (std_g + eps)

    # 面试高频考点：为什么要 detach？
    # 答：advantages 来自外部奖励，是"标量信号"而非可微分的计算图
    # 如果不 detach，梯度会流经 rewards（虽然 rewards 通常不需要梯度）
    # detach() 确保梯度只通过 logps 流动，实现纯粹的策略梯度更新
    advantages_detached = advantages.detach()

    # 策略梯度目标：max E[A * log π]  →  min -E[A * log π]
    return -(advantages_detached * logps).mean()

In [ ]:
# Demo
logps = torch.tensor([0.0, -0.5, -1.0, -1.5])
rewards = torch.tensor([1.0, 0.8, 0.2, 0.0])
group_ids = torch.tensor([0, 0, 1, 1])
print('Loss:', grpo_loss(logps, rewards, group_ids).item())

In [ ]:
from torch_judge import check
check('grpo_loss')

## 📝 核心思路总结

1. **GRPO = 组内相对优势策略优化**：同一 prompt 的多个回复互相比较，而非跨 prompt
2. **z-score 归一化消除组间奖励尺度差异**：不同 prompt 的奖励可能差异很大，归一化后统一尺度
3. **advantages.detach() 是关键**：梯度只通过 logps 流动，奖励被视为常量